# 02 - Train One Registered Alignment Run

[Open in Google Colab](https://colab.research.google.com/github/zhaoqyu/Lab-NLP/blob/mike/alignment_benchmark/notebooks/02_train_one_run.ipynb)

**Objective.** Fit or resume one QLoRA control/intervention adapter in a Colab-safe unit of work.

Use a GPU runtime. Persistent artifacts are written to Google Drive, so interrupted Colab sessions can resume. Run cells from top to bottom.

In [ ]:
import os
import subprocess
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive')
assert subprocess.run(['nvidia-smi'], check=False).returncode == 0, 'Select a GPU runtime first.'


In [ ]:
REPO_URL = 'https://github.com/zhaoqyu/Lab-NLP.git'
BRANCH = 'mike'
REPO_DIR = Path('/content/Lab-NLP')

if not (REPO_DIR / '.git').exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'checkout', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO_DIR, check=True)

os.chdir(REPO_DIR)
print('Repository:', REPO_DIR)


In [ ]:
subprocess.run(
    ['python', '-m', 'pip', 'install', '-q', '-r', 'alignment_benchmark/requirements-colab.txt'],
    check=True,
)
subprocess.run(
    ['python', '-m', 'pip', 'install', '-q', '-e', './alignment_benchmark', '--no-deps'],
    check=True,
)


In [ ]:
OUTPUT_ROOT = Path('/content/drive/MyDrive/Lab-NLP/valuebench-paper')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
os.environ['VALUEBENCH_OUTPUT_ROOT'] = str(OUTPUT_ROOT)
CONFIG = 'alignment_benchmark/configs/paper.yaml'

def run(*arguments: str) -> None:
    command = ['valuebench', *arguments, '--config', CONFIG]
    print('Running:', ' '.join(command))
    subprocess.run(command, check=True)

print('Persistent output:', OUTPUT_ROOT)


In [ ]:
run('doctor', '--strict')


## Select one registered run

Start with controls. A target evaluation requires the control adapter for the same method and seed. Valid methods are `sft`, `dpo`, `hypo`, `ipo`, `simpo`, and `orpo`.

In [ ]:
METHOD = 'dpo'
TARGET = 'control'  # or one of the ten basic values, e.g. Security
SEED = 13

METHOD, TARGET, SEED


In [ ]:
run(
    'train',
    '--method', METHOD,
    '--target', TARGET,
    '--seed', str(SEED),
)


## Inspect progress

The trainer resumes the last checkpoint if the runtime stopped. A completed run has a final adapter, manifest, trainer state, TensorBoard logs, and `DONE` marker.

In [ ]:
run('status')


## Queue-style alternative

Set `USE_REGISTERED_QUEUE = True` to run exactly one ready evaluation or pending construction job. Rerun the notebook in later sessions until `status` is complete.

In [ ]:
USE_REGISTERED_QUEUE = False
if USE_REGISTERED_QUEUE:
    run('run-next')
